In [1]:
# Configuration
TARGET_COLUMN = 'BBSE3.SA'  # Stock ticker for Viatris

# Step 1: Import Libraries and Install Dependencies
print("\nStep 1: Installing dependencies...")
!pip install --force-reinstall tensorflow==2.16.1 numpy==2.0.2 pandas==2.2.2 scikit-learn==1.5.2 optuna==4.0.0 lightgbm==4.5.0 xgboost==2.1.1 pywt==1.5.0 pykalman==0.9.7 transformers==4.44.2 torch==2.4.1 requests==2.32.3 beautifulsoup4==4.12.3
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # For CUDA debugging
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from google.colab import files
import warnings
warnings.filterwarnings('ignore')
import time
import random
import torch


Step 1: Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 5.3 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement pywt==1.5.0 (from versions: none)
ERROR: No matching distribution found for pywt==1.5.0


In [2]:
# Optuna
try:
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True
    print("Optuna version:", optuna.__version__)
except Exception:
    print("Optuna installation failed.")
    !pip install optuna==4.0.0
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True

# TensorFlow
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Conv1D, GRU, Dropout, MultiHeadAttention, GlobalAveragePooling1D, Input, Flatten
    from tensorflow.keras.regularizers import l2
    TF_AVAILABLE = True
    print("TensorFlow version:", tf.__version__)
except Exception:
    print("TensorFlow installation failed.")
    !pip install tensorflow==2.16.1
    import tensorflow as tf
    TF_AVAILABLE = True

# PyWavelets
try:
    import pywt
    PYWT_AVAILABLE = True
except Exception:
    print("PyWavelets installation failed.")
    !pip install pywt==1.5.0
    import pywt
    PYWT_AVAILABLE = True

# PyKalman
try:
    from pykalman import KalmanFilter
    PYKALMAN_AVAILABLE = True
except Exception:
    print("PyKalman installation failed.")
    !pip install pykalman==0.9.7
    from pykalman import KalmanFilter
    PYKALMAN_AVAILABLE = True

# LightGBM
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except Exception:
    LGB_AVAILABLE = False

# XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

# Transformers and PyTorch
TRANSFORMERS_AVAILABLE = False
try:
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    TRANSFORMERS_AVAILABLE = True
    print("Transformers version:", transformers.__version__)
    print("PyTorch version:", torch.__version__)
except Exception:
    print("Transformers or PyTorch installation failed.")
    !pip install transformers==4.44.2 torch==2.4.1
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    TRANSFORMERS_AVAILABLE = True

# Requests and BeautifulSoup
import requests
from bs4 import BeautifulSoup

Optuna installation failed.
  Using cached optuna-4.0.0-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.8/362.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 24.4 MB/s eta 0:00:00
TensorFlow version: 2.18.0
PyKalman installation failed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 17.6 MB/s eta 0:00:00
Transformers version: 4.52.4
PyTorch version: 2.6.0+cu124


In [3]:
# Step 2: Load and Clean Data
print("\nStep 2: Loading and cleaning data...")
try:
    uploaded = files.upload()
    stock_data = {}
    news_data = None
    for file_name in uploaded.keys():
        print(f"Processing file: {file_name}")
        if file_name.endswith('.csv'):
            if 'news_data' in file_name.lower():
                try:
                    news_data = pd.read_csv(file_name)
                    print(f"News data loaded: {len(news_data)} rows")
                    print("News data columns:", news_data.columns.tolist())
                    print("News data sample:\n", news_data.head(2))
                except Exception as e:
                    print(f"Error loading news data {file_name}: {e}")
                    news_data = None
            else:
                try:
                    df = pd.read_csv(file_name)
                    print(f"Columns: {df.columns.tolist()}, Rows: {len(df)}")
                    # Check for both 'Timestamp' and 'Timestamp '
                    timestamp_col = 'Timestamp' if 'Timestamp' in df.columns else 'Timestamp ' if 'Timestamp ' in df.columns else None
                    if timestamp_col is None or TARGET_COLUMN not in df.columns:
                        raise ValueError(f"Expected 'Timestamp' or 'Timestamp ' and '{TARGET_COLUMN}' columns")
                    stock_data[TARGET_COLUMN] = df
                    stock_data[TARGET_COLUMN].rename(columns={timestamp_col: 'Timestamp'}, inplace=True) # Rename to 'Timestamp'
                    stock_data[TARGET_COLUMN][TARGET_COLUMN] = stock_data[TARGET_COLUMN][TARGET_COLUMN].interpolate().ffill().bfill()
                    stock_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].dropna()
                    stock_data[TARGET_COLUMN]['Timestamp'] = pd.to_datetime(stock_data[TARGET_COLUMN]['Timestamp'], errors='coerce')
                    stock_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].dropna(subset=['Timestamp'])
                    print(f"Rows after cleaning: {len(stock_data[TARGET_COLUMN])}")
                    print("Price data sample:\n", stock_data[TARGET_COLUMN].head(2))
                except Exception as e:
                    print(f"Error loading price data {file_name}: {e}")
except Exception as e:
    print(f"Error during file upload: {e}")
    exit()


Step 2: Loading and cleaning data...


Saving BBSE3.SA_5.csv to BBSE3.SA_5.csv
Processing file: BBSE3.SA_5.csv
Columns: ['Timestamp', 'BBSE3.SA'], Rows: 8500
Rows after cleaning: 8500
Price data sample:
                   Timestamp   BBSE3.SA
0 2025-05-27 13:03:00+00:00  38.150002
1 2025-05-27 13:04:00+00:00  38.200001


In [4]:
# Step 2.5: Fetch News from FinViz if No News Data Provided
if news_data is None and TRANSFORMERS_AVAILABLE:
    print("\nStep 2.5: No news data uploaded. Fetching news from FinViz...")
    def fetch_finviz_news(ticker):
        url = f'https://finviz.com/quote.ashx?t={ticker}'
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
        }
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            news_table = soup.find('table', {'id': 'news-table'})
            if not news_table:
                print("No news table found on FinViz")
                return None
            news_data = []
            for row in news_table.findAll('tr'):
                cells = row.findAll('td')
                if len(cells) < 2:
                    continue
                headline_link = cells[1].find('a')
                if not headline_link:
                    continue
                headline = headline_link.text.strip()
                link = headline_link['href']
                if link.startswith('/'):
                    link = f'https://finviz.com{link}'
                news_data.append({'Headline': headline, 'Headline_Link': link})
                time.sleep(0.5)
            df = pd.DataFrame(news_data)
            df = df.head(100)  # Limit to last 100 articles
            df['News_Index'] = range(1, len(df) + 1)  # 1 (least recent) to 100 (most recent)
            print(f"Fetched {len(df)} news articles from FinViz")
            return df
        except Exception as e:
            print(f"Error fetching FinViz data for {ticker}: {e}")
            return None

    def fetch_article_content(url, headline):
        user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36'
        ]
        for attempt in range(3):
            try:
                headers = {'User-Agent': random.choice(user_agents)}
                response = requests.get(url, headers=headers, timeout=10)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, 'html.parser')
                paragraphs = soup.find_all('p')
                article_text = ' '.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])
                if len(article_text.strip()) > 50:
                    print(f"Successfully fetched article from {url[:50]}...")
                    return article_text
                else:
                    print(f"Short content from {url[:50]}...")
                    return headline
            except Exception as e:
                print(f"Error fetching article from {url[:50]}...: {e}")
                if attempt < 2:
                    time.sleep(2 ** attempt)
                else:
                    return headline
        return headline

    news_data = fetch_finviz_news(TARGET_COLUMN)
    if news_data is not None and not news_data.empty:
        print(f"Fetching article content for {len(news_data)} articles...")
        news_data['News_detail'] = news_data.apply(lambda row: fetch_article_content(row['Headline_Link'], row['Headline']), axis=1)
        news_data['News_detail'] = news_data['News_detail'].apply(lambda x: str(x) if x and len(str(x).strip()) > 0 else None)
        valid_articles = news_data['News_detail'].notna().sum()
        print(f"Fetched {valid_articles} valid articles out of {len(news_data)}")
        if valid_articles == 0:
            print("No valid article content fetched. Disabling sentiment analysis.")
            news_data = None
    else:
        print("Failed to fetch news from FinViz.")


Step 2.5: No news data uploaded. Fetching news from FinViz...
Error fetching FinViz data for BBSE3.SA: 404 Client Error: Not Found for url: https://finviz.com/quote.ashx?t=BBSE3.SA
Failed to fetch news from FinViz.


In [5]:
# Step 2.6: Process News Data with FinBERT
if TRANSFORMERS_AVAILABLE and news_data is not None:
    print("\nStep 2.6: Processing news data with FinBERT...")
    tokenizer = None
    model = None
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
    try:
        torch.cuda.empty_cache()
        tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        model.to(device)
        print(f"FinBERT model loaded on {device}.")
    except Exception as e:
        print(f"Failed to load FinBERT: {e}")
        TRANSFORMERS_AVAILABLE = False
        news_data = None

    if news_data is not None:
        print("Summarizing news articles...")
        summarizer = None
        try:
            torch.cuda.empty_cache()
            # Using device=-1 for CPU, which is the default for summarization pipeline
            summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)
            print("Summarizer loaded on CPU.")
        except Exception as e:
            print(f"Failed to load summarizer: {e}")
            # Fallback to using headline if summarizer fails
            news_data['Summary'] = news_data['News_detail'].fillna(news_data['Headline'])

        if summarizer:
            def summarize_text(text):
                if not text or pd.isna(text) or len(str(text).strip()) == 0:
                    return ""
                text = str(text)[:1024]  # Truncate to avoid exceeding max length
                word_count = len(str(text).split())
                if word_count <= 30:  # Summarize only if text long enough
                    return text
                try:
                    # Adjust max_length and min_length based on input length
                    max_len = min(150, max(30, int(word_count * 0.3)))
                    min_len = max(10, min(max_len, int(word_count * 0.1)))
                    summary = summarizer(text, max_length=max_len, min_length=min_len, do_sample=False, batch_size=1)[0]['summary_text']
                    return summary
                except Exception as e:
                    print(f"Error in summarization for text {text[:50]}...: {e}")
                    return text[:256]  # Return truncated text on failure

            # Apply summarization only to rows where News_detail is not None and not empty
            news_data['Summary'] = news_data.apply(
                lambda row: summarize_text(row['News_detail']) if pd.notna(row['News_detail']) and len(str(row['News_detail']).strip()) > 0 else row['Headline'],
                axis=1
            )

            unique_summaries = len(news_data['Summary'].unique())
            print(f"\nGenerated {unique_summaries} unique summaries")
            print(f"News summaries:\n", news_data[['Summary']].head(5))

        def get_finbert_sentiment(text):
            if not text or pd.isna(text) or len(str(text).strip()) == 0:
                return np.array([0.33, 0.33, 0.33])  # Return neutral default scores for invalid text
            try:
                # Use the text directly for sentiment analysis, truncation handled by tokenizer
                inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
                with torch.no_grad():
                    outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
                return probs.cpu().numpy().flatten()
            except Exception as e:
                print(f"Error analyzing sentiment for text {str(text)[:50]}...: {e}")
                return np.array([0.33, 0.33, 0.33])  # Return neutral scores on failure

        # Apply sentiment analysis to the 'Summary' column
        news_data['sentiment_scores'] = news_data['Summary'].apply(
            lambda x: get_finbert_sentiment(x)
        )

        # Expand sentiment scores into separate columns
        sentiment_scores_list = news_data['sentiment_scores'].tolist()
        sentiment_scores_df = pd.DataFrame(sentiment_scores_list, index=news_data.index)
        news_data[['positive', 'negative', 'neutral']] = sentiment_scores_df

        news_data['news_polarity'] = news_data['positive'] - news_data['negative']
        news_data['news_strength'] = news_data[['positive', 'negative', 'neutral']].max(axis=1)
        news_data['news_weight'] = news_data['News_Index'] / news_data['News_Index'].max() if news_data['News_Index'].max() > 0 else 0  # Normalize weight
        news_data['news_impact'] = news_data['news_polarity'] * news_data['news_strength'] * news_data['news_weight']
        print(f"Sentiment polarity and impact:\n", news_data[['News_Index', 'positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_weight', 'news_impact']].head())

        # Merge news impact data with price data
        try:
            if TARGET_COLUMN in stock_data and len(stock_data[TARGET_COLUMN]) > 0:
                print(f"Aligning news data with price data...")
                stock_df = stock_data[TARGET_COLUMN].sort_values(by='Timestamp').reset_index(drop=True)
                news_data_aligned = news_data.reset_index(drop=True)  # Ensure index is reset

                # Ensure news_data_aligned has same number of rows as stock_df for direct assignment
                if len(news_data_aligned) > len(stock_df):
                    news_data_aligned = news_data_aligned.iloc[:len(stock_df)]
                elif len(news_data_aligned) < len(stock_df):
                    # If news data is shorter, replicate the last row to match the length
                    last_row = news_data_aligned.iloc[-1]
                    filler_rows = pd.DataFrame([last_row] * (len(stock_df) - len(news_data_aligned)))
                    news_data_aligned = pd.concat([news_data_aligned, filler_rows], ignore_index=True)

                # Add news sentiment columns to stock_df
                news_sentiment_cols = ['positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_weight', 'news_impact']
                if all(col in news_data_aligned.columns for col in news_sentiment_cols):
                    merged_data = stock_df.copy()
                    # Assign the sentiment columns directly as both dataframes have same length and index
                    merged_data[news_sentiment_cols] = news_data_aligned[news_sentiment_cols]
                    # Fill potential NaN with neutral sentiment
                    merged_data[news_sentiment_cols] = merged_data[news_sentiment_cols].fillna(0.33)  # Assuming 0.33 is neutral for probability-based scores
                    stock_data[TARGET_COLUMN] = merged_data
                    print(f"Merged data shape: {stock_data[TARGET_COLUMN].shape}")
                    print(f"Merged sample:\n", stock_data[TARGET_COLUMN].head(5))
                else:
                    print("Sentiment columns not found in news_data after alignment. Proceeding without sentiment features.")
            else:
                print(f"No valid stock data or empty. Exiting...")
                exit()
        except Exception as e:
            print(f"Error during merging news data: {e}")
            print("Proceeding without sentiment features...")
            news_data = None
else:
    print("No news_data available or Transformers unavailable.")

No news_data available or Transformers unavailable.


In [6]:
# Step 3: Denoise Data for Highest SNR
print("\nStep 3: Denoising data...")
price_scaler = MinMaxScaler()
feature_columns = [TARGET_COLUMN]
if TRANSFORMERS_AVAILABLE and news_data is not None:
    feature_columns.extend(['positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_impact'])

stock_denoised_data = {} # Initialize stock_denoised_data dictionary

if TARGET_COLUMN in stock_data and not stock_data[TARGET_COLUMN].empty:
    scaled_data = price_scaler.fit_transform(stock_data[TARGET_COLUMN][[TARGET_COLUMN]])
    print(f"Scaled data shape: {scaled_data.shape}")

    def calculate_snr(original, denoised):
        noise = original - denoised
        signal_power = np.mean(original ** 2)
        noise_power = np.var(noise)
        return 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float('inf')

    def optimize_denoising(trial, data):
        methods = []
        if PYWT_AVAILABLE:
            wavelet = trial.suggest_categorical('wavelet', ['db1', 'db2', 'sym2'])
            level = trial.suggest_int('wavelet_level', 1, 3)
            wavelet_denoised = pywt.wavedec(data, wavelet, level=level)
            threshold = np.std(wavelet_denoised[-1]) * np.sqrt(2 * np.log(len(data)))
            wavelet_denoised = pywt.waverec([pywt.threshold(c, threshold, mode='soft') for c in wavelet_denoised], wavelet)[:len(data)]
            snr_wavelet = calculate_snr(data, wavelet_denoised)
            methods.append(('wavelet', snr_wavelet, wavelet_denoised))
        ma_window = trial.suggest_int('ma_window', 3, 10)
        ma_denoised = pd.Series(data).rolling(window=ma_window, min_periods=1).mean().values
        snr_ma = calculate_snr(data, ma_denoised)
        methods.append(('ma', snr_ma, ma_denoised))
        if TF_AVAILABLE:
            encoding_dim = trial.suggest_int('encoding_dim', 16, 32)
            model = Sequential([Dense(encoding_dim, activation='relu', input_shape=(1,)),
                                Dense(1, activation='linear')])
            model.compile(optimizer='adam', loss='mse')
            autoencoder_denoised = model.predict(data.reshape(-1, 1), verbose=0).flatten()
            snr_autoencoder = calculate_snr(data, autoencoder_denoised)
            methods.append(('autoencoder', snr_autoencoder, autoencoder_denoised))
        if PYKALMAN_AVAILABLE:
            kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1], initial_state_mean=data[0])
            kalman_denoised, _ = kf.filter(data)
            snr_kalman = calculate_snr(data, kalman_denoised.flatten())
            methods.append(('kalman', snr_kalman, kalman_denoised))
        best_method = max(methods, key=lambda x: x[1] if not np.isinf(x[1]) else -float('inf'))
        return best_method[1], best_method[2]

    if OPTUNA_AVAILABLE:
        print("Optimizing denoising with Optuna...")
        study_denoise = optuna.create_study(direction='maximize', sampler=TPESampler())
        study_denoise.optimize(lambda trial: optimize_denoising(trial, scaled_data[:, 0])[0], n_trials=5)
        _, denoised_data = optimize_denoising(optuna.trial.FixedTrial(study_denoise.best_params), scaled_data[:, 0])
        print(f"Best SNR after Optuna: {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")
    else:
        print("Optuna unavailable. Using default moving average denoising...")
        denoised_data = pd.Series(scaled_data[:, 0]).rolling(window=5, min_periods=1).mean().values
        print(f"Best SNR (default): {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")

    # Ensure stock_denoised_data is populated
    stock_denoised_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].copy()
    stock_denoised_data[TARGET_COLUMN][TARGET_COLUMN] = denoised_data

    if TRANSFORMERS_AVAILABLE and news_data is not None and all(col in news_data.columns for col in ['positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_impact']):
         # Align and merge news sentiment data with denoised stock data
        news_sentiment_cols = ['positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_impact']
        # Ensure news_data_aligned has same number of rows as stock_data[TARGET_COLUMN] for direct assignment
        if len(news_data) > len(stock_data[TARGET_COLUMN]):
            news_data_aligned = news_data.iloc[:len(stock_data[TARGET_COLUMN])].reset_index(drop=True)
        elif len(news_data) < len(stock_data[TARGET_COLUMN]):
            last_row = news_data.iloc[-1]
            filler_rows = pd.DataFrame([last_row] * (len(stock_data[TARGET_COLUMN]) - len(news_data))).reset_index(drop=True)
            news_data_aligned = pd.concat([news_data.reset_index(drop=True), filler_rows], ignore_index=True)
        else:
            news_data_aligned = news_data.reset_index(drop=True)

        stock_denoised_data[TARGET_COLUMN][news_sentiment_cols] = news_data_aligned[news_sentiment_cols]
        stock_denoised_data[TARGET_COLUMN][news_sentiment_cols] = stock_denoised_data[TARGET_COLUMN][news_sentiment_cols].fillna(0.33) # Fill NaN with neutral

    print(f"Stock denoised data shape: {stock_denoised_data[TARGET_COLUMN].shape}")
    print(f"Stock denoised data sample:\n", stock_denoised_data[TARGET_COLUMN].head(5))

else:
    print("No valid stock data available after cleaning. Cannot proceed with denoising.")
    # Ensure stock_denoised_data is still an empty dict if no data
    stock_denoised_data = {}

[I 2025-06-29 18:06:54,621] A new study created in memory with name: no-name-e19c0698-ccdd-4053-ba4a-58134e8349a2



Step 3: Denoising data...
Scaled data shape: (8500, 1)
Optimizing denoising with Optuna...


[I 2025-06-29 18:07:08,530] Trial 0 finished with value: 45.79042253951238 and parameters: {'wavelet': 'db2', 'wavelet_level': 3, 'ma_window': 5, 'encoding_dim': 29}. Best is trial 0 with value: 45.79042253951238.
[I 2025-06-29 18:07:19,739] Trial 1 finished with value: 45.79042253951238 and parameters: {'wavelet': 'db2', 'wavelet_level': 2, 'ma_window': 4, 'encoding_dim': 16}. Best is trial 0 with value: 45.79042253951238.
[I 2025-06-29 18:07:29,529] Trial 2 finished with value: 45.79042253951238 and parameters: {'wavelet': 'db1', 'wavelet_level': 3, 'ma_window': 8, 'encoding_dim': 28}. Best is trial 0 with value: 45.79042253951238.
[I 2025-06-29 18:07:34,215] Trial 3 finished with value: 45.79042253951238 and parameters: {'wavelet': 'sym2', 'wavelet_level': 3, 'ma_window': 8, 'encoding_dim': 31}. Best is trial 0 with value: 45.79042253951238.
[I 2025-06-29 18:07:38,738] Trial 4 finished with value: 45.79042253951238 and parameters: {'wavelet': 'db2', 'wavelet_level': 1, 'ma_window': 

Best SNR after Optuna: 1.24 dB
Stock denoised data shape: (8500, 2)
Stock denoised data sample:
                   Timestamp  BBSE3.SA
0 2025-05-27 13:03:00+00:00  0.865790
1 2025-05-27 13:04:00+00:00  0.873685
2 2025-05-27 13:05:00+00:00  0.880162
3 2025-05-27 13:06:00+00:00  0.890790
4 2025-05-27 13:07:00+00:00  0.888350


In [7]:
# Step 4: Prepare Sequences
print("\nStep 4: Preparing sequences...")
SEQ_LENGTH = 30
WINDOW_SIZE = 300
stock_X_deep = {}
stock_X_ml = {}
stock_y = {}
full_data = stock_denoised_data[TARGET_COLUMN][feature_columns].values
X_for_prediction = np.array([full_data[i:i + SEQ_LENGTH] for i in range(len(full_data) - SEQ_LENGTH)])
stock_y[TARGET_COLUMN] = stock_denoised_data[TARGET_COLUMN][TARGET_COLUMN].values[SEQ_LENGTH:]
stock_X_deep[TARGET_COLUMN] = X_for_prediction
stock_X_ml[TARGET_COLUMN] = X_for_prediction
print(f"X_deep shape: {stock_X_deep[TARGET_COLUMN].shape}, y shape: {stock_y[TARGET_COLUMN].shape}")
last_300_scaled = stock_denoised_data[TARGET_COLUMN][feature_columns].values[-WINDOW_SIZE:]


Step 4: Preparing sequences...
X_deep shape: (8470, 30, 1), y shape: (8470,)


In [8]:
# Step 5: Define and Optimize Models
print("\nStep 5: Defining and optimizing models...")
feature_dim = len(feature_columns)
def optimize_cnn_lstm(trial):
    filters = trial.suggest_int('filters', 32, 64)
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_conv1d_lstm(trial):
    filters = trial.suggest_int('filters', 32, 64)
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_bilstm(trial):
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Bidirectional(LSTM(lstm_units, kernel_regularizer=l2(0.02))),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_stacked_lstm(trial):
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_gru(trial):
    gru_units = trial.suggest_int('gru_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        GRU(gru_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_lightgbm(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-5, 1.0)
    }
    return lgb.LGBMRegressor(**params, force_col_wise=True)

def optimize_xgboost(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0)
    }
    return xgb.XGBRegressor(**params)

def optimize_transformer(trial):
    num_heads = trial.suggest_int('num_heads', 4, 8)
    ff_dim = trial.suggest_int('ff_dim', 64, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(inputs, inputs)
    x = Dropout(dropout_rate)(x)
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(x, x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(ff_dim, activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_tcn(trial):
    filters = trial.suggest_int('filters', 32, 64)
    kernel_size = trial.suggest_int('kernel_size', 2, 5)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = inputs
    shortcut = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    shortcut = Conv1D(filters=filters, kernel_size=1, padding='same')(shortcut)
    x = tf.keras.layers.Add()([shortcut, x])
    x = tf.keras.layers.Activation('relu')(x)
    x = Flatten()(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(0.02))(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model


Step 5: Defining and optimizing models...


In [9]:
models = [
    ('cnn_lstm', optimize_cnn_lstm, 'deep'),
    ('conv1d_lstm', optimize_conv1d_lstm, 'deep'),
    ('bilstm', optimize_bilstm, 'deep'),
    ('stacked_lstm', optimize_stacked_lstm, 'deep'),
    ('gru', optimize_gru, 'deep'),
    ('lightgbm', optimize_lightgbm, 'ml'),
    ('xgboost', optimize_xgboost, 'ml'),
    ('transformer', optimize_transformer, 'deep'),
    ('tcn', optimize_tcn, 'deep')
]

# Step 6: Train and Evaluate Models with Walk-Forward Validation
print("\nStep 6: Training and evaluating models...")
def walk_forward_validation(model, model_name, X, y, price_scaler, step_size=300, max_steps=20):
    best_adj_r2 = -float('inf')
    best_model = None
    adj_r2_history = []
    n_steps = min(max_steps, max(1, (len(X) - SEQ_LENGTH) // step_size))
    if n_steps < 1:
        print(f"Error: Insufficient data for walk-forward validation for {model_name}.")
        return {'adj_r2': 0, 'best_model': model}, adj_r2_history
    n_predictors = feature_dim
    for i in range(1, n_steps + 1):
        train_end = i * step_size
        test_size = min(step_size, len(X) - train_end)
        train_X = X[:train_end]
        train_y = y[:train_end]
        test_X = X[train_end:train_end + test_size]
        test_y = y[train_end:train_end + test_size]
        if len(test_X) == 0 or len(test_y) == 0 or len(test_X) != len(test_y):
            continue
        try:
            if model_name in ['xgboost', 'lightgbm']:
                model.fit(train_X[:, :, 0].reshape(-1, SEQ_LENGTH), train_y)
                preds = model.predict(test_X[:, :, 0].reshape(-1, SEQ_LENGTH))
            else:
                model.fit(train_X, train_y, epochs=10, batch_size=64, validation_split=0.2, verbose=0,
                          callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
                preds = model.predict(test_X, verbose=0).flatten()
            test_y_inv = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
            preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
            ss_res = np.sum((test_y_inv - preds_inv) ** 2)
            ss_tot = np.sum((test_y_inv - np.mean(test_y_inv)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
            n = len(test_y_inv)
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_predictors - 1) if n > n_predictors + 1 else 0
            adj_r2_history.append(adj_r2)
            if adj_r2 > best_adj_r2:
                best_adj_r2 = adj_r2
                best_model = tf.keras.models.clone_model(model) if model_name not in ['xgboost', 'lightgbm'] else model
                if model_name not in ['xgboost', 'lightgbm']:
                    best_model.set_weights(model.get_weights())
        except Exception as e:
            print(f"Error in walk-forward validation for {model_name} at step {i}: {e}")
            continue
    return {'adj_r2': best_adj_r2, 'best_model': best_model}, adj_r2_history


Step 6: Training and evaluating models...


In [10]:
stock_results = {}
for stock_name in stock_data.keys():
    results = {}
    for model_name, model_fn, data_type in models:
        if not (TF_AVAILABLE and data_type == 'deep') and not (LGB_AVAILABLE and model_name == 'lightgbm') and not (XGB_AVAILABLE and model_name == 'xgboost'):
            continue
        print(f"\nTraining {model_name} for {stock_name}...")
        X_data = stock_X_deep[stock_name] if data_type == 'deep' else stock_X_ml[stock_name]
        try:
            if OPTUNA_AVAILABLE:
                study = optuna.create_study(direction='minimize', sampler=TPESampler())
                def objective(trial):
                    model = model_fn(trial)
                    n_samples = len(X_data)
                    train_size = int(0.8 * n_samples)
                    train_X, train_y = X_data[:train_size], stock_y[stock_name][:train_size]
                    val_X, val_y = X_data[train_size:], stock_y[stock_name][train_size:]
                    if model_name in ['xgboost', 'lightgbm']:
                        model.fit(train_X[:, :, 0].reshape(-1, SEQ_LENGTH), train_y)
                        preds = model.predict(val_X[:, :, 0].reshape(-1, SEQ_LENGTH))
                    else:
                        model.fit(train_X, train_y, epochs=5, batch_size=64, validation_split=0.2, verbose=0,
                                  callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)])
                        preds = model.predict(val_X, verbose=0).flatten()
                    val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
                    preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
                    ss_res = np.sum((val_y_inv - preds_inv) ** 2)
                    ss_tot = np.sum((val_y_inv - np.mean(val_y_inv)) ** 2)
                    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
                    n = len(val_y_inv)
                    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
                    return -adj_r2
                study.optimize(objective, n_trials=3)
                model = model_fn(optuna.trial.FixedTrial(study.best_params))
            else:
                model = model_fn({})
            result, adj_r2_history = walk_forward_validation(model, model_name, X_data, stock_y[stock_name], price_scaler)
            results[model_name] = result
            print(f"{model_name} - {stock_name} - Best Adj R²: {result['adj_r2']:.6f}")
        except Exception as e:
            print(f"Error training {model_name} for {stock_name}: {e}")
            results[model_name] = {'adj_r2': 0, 'best_model': None}
    stock_results[stock_name] = results

[I 2025-06-29 18:08:37,334] A new study created in memory with name: no-name-fba94e82-be6a-4f6c-a0c7-4ceae745a325



Training cnn_lstm for BBSE3.SA...


[I 2025-06-29 18:08:55,299] Trial 0 finished with value: 1.128024228571522 and parameters: {'filters': 54, 'lstm_units': 98, 'dropout_rate': 0.568171817168728, 'learning_rate': 0.0032099396115482493}. Best is trial 0 with value: 1.128024228571522.
[I 2025-06-29 18:09:16,469] Trial 1 finished with value: -0.6467571569010264 and parameters: {'filters': 62, 'lstm_units': 94, 'dropout_rate': 0.5142456114814248, 'learning_rate': 0.001960190117725967}. Best is trial 1 with value: -0.6467571569010264.
[I 2025-06-29 18:09:31,239] Trial 2 finished with value: -0.7173254005216643 and parameters: {'filters': 34, 'lstm_units': 57, 'dropout_rate': 0.3244697527795214, 'learning_rate': 0.003564741579517217}. Best is trial 2 with value: -0.7173254005216643.
[I 2025-06-29 18:12:36,670] A new study created in memory with name: no-name-69b7df3e-96a9-4dd9-b82b-dfd272fb8012


cnn_lstm - BBSE3.SA - Best Adj R²: 0.013795

Training conv1d_lstm for BBSE3.SA...


[I 2025-06-29 18:12:46,646] Trial 0 finished with value: -0.13673817319630388 and parameters: {'filters': 53, 'lstm_units': 122, 'dropout_rate': 0.3340275856050143, 'learning_rate': 0.0035495472892141805}. Best is trial 0 with value: -0.13673817319630388.
[I 2025-06-29 18:12:55,008] Trial 1 finished with value: -0.7456376030563103 and parameters: {'filters': 52, 'lstm_units': 99, 'dropout_rate': 0.3909287644847751, 'learning_rate': 0.004419501918025894}. Best is trial 1 with value: -0.7456376030563103.
[I 2025-06-29 18:13:03,502] Trial 2 finished with value: -0.7449292569377731 and parameters: {'filters': 50, 'lstm_units': 84, 'dropout_rate': 0.5059442072177036, 'learning_rate': 0.00292247085649728}. Best is trial 1 with value: -0.7456376030563103.
[I 2025-06-29 18:15:20,079] A new study created in memory with name: no-name-3475b782-6cc8-4a37-b10e-d4594ea27127


...

xgboost - BBSE3.SA - Best Adj R²: 0.989604

Training transformer for BBSE3.SA...


[I 2025-06-29 18:30:28,348] Trial 0 finished with value: -0.4220253077180123 and parameters: {'num_heads': 8, 'ff_dim': 107, 'dropout_rate': 0.5416081372045882, 'learning_rate': 0.0010111735301737246}. Best is trial 0 with value: -0.4220253077180123.
[I 2025-06-29 18:30:43,007] Trial 1 finished with value: -0.44975915756229745 and parameters: {'num_heads': 6, 'ff_dim': 120, 'dropout_rate': 0.4123864779464418, 'learning_rate': 0.0038561508357837417}. Best is trial 1 with value: -0.44975915756229745.
[I 2025-06-29 18:31:02,491] Trial 2 finished with value: -0.7183017727248315 and parameters: {'num_heads': 4, 'ff_dim': 75, 'dropout_rate': 0.4550874357757878, 'learning_rate': 0.0017047757098648723}. Best is trial 2 with value: -0.7183017727248315.
[I 2025-06-29 18:32:22,585] A new study created in memory with name: no-name-7f9bb056-50fc-45a2-8ca8-bcbc46a8d496


transformer - BBSE3.SA - Best Adj R²: 0.467023

Training tcn for BBSE3.SA...


[I 2025-06-29 18:32:37,610] Trial 0 finished with value: -0.898118823167164 and parameters: {'filters': 56, 'kernel_size': 2, 'dropout_rate': 0.5411395926571368, 'learning_rate': 0.0009137207358597265}. Best is trial 0 with value: -0.898118823167164.
[I 2025-06-29 18:32:48,646] Trial 1 finished with value: -0.8871424368405219 and parameters: {'filters': 50, 'kernel_size': 4, 'dropout_rate': 0.47531068721833936, 'learning_rate': 0.002642316012114385}. Best is trial 0 with value: -0.898118823167164.
[I 2025-06-29 18:33:03,315] Trial 2 finished with value: -0.7679912879297417 and parameters: {'filters': 56, 'kernel_size': 5, 'dropout_rate': 0.5068199069826815, 'learning_rate': 0.0049931818616498706}. Best is trial 0 with value: -0.898118823167164.


tcn - BBSE3.SA - Best Adj R²: 0.921833


In [11]:
# Step 6.5: Analyze Overfitting with Validation and Test Sets
print("\nStep 6.5: Analyzing overfitting with validation and test sets...")
for stock_name in stock_results:
    results = stock_results[stock_name]
    X_data = stock_X_deep[stock_name] if 'cnn_lstm' in results else stock_X_ml[stock_name]
    y_data = stock_y[stock_name]
    n_samples = len(X_data)

    # Split data: last 150 for test, remaining for train+val
    test_size = 150
    if n_samples <= test_size:
        print(f"Error: Not enough samples for {stock_name}. Need > {test_size} samples, got {n_samples}")
        continue
    train_val_size = n_samples - test_size
    train_size = int(0.8 * train_val_size)  # 80% of train+val for training
    val_size = train_val_size - train_size

    train_X, val_X, test_X = X_data[:train_size], X_data[train_size:train_val_size], X_data[train_val_size:]
    train_y, val_y, test_y = y_data[:train_size], y_data[train_size:train_val_size], y_data[train_val_size:]

    for model_name, result in results.items():
        best_model = result['best_model']
        if best_model is None:
            print(f"No model found for {model_name} in {stock_name}")
            continue
        try:
            if model_name in ['xgboost', 'lightgbm']:
                train_pred = best_model.predict(train_X[:, :, 0].reshape(-1, SEQ_LENGTH))
                val_pred = best_model.predict(val_X[:, :, 0].reshape(-1, SEQ_LENGTH))
                test_pred = best_model.predict(test_X[:, :, 0].reshape(-1, SEQ_LENGTH))
            else:
                best_model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
                best_model.fit(train_X, train_y, epochs=5, batch_size=64, verbose=0,
                              callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)])
                train_pred = best_model.predict(train_X, verbose=0).flatten()
                val_pred = best_model.predict(val_X, verbose=0).flatten()
                test_pred = best_model.predict(test_X, verbose=0).flatten()

            train_y_inv = price_scaler.inverse_transform(train_y.reshape(-1, 1)).flatten()
            val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
            test_y_inv = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
            train_pred_inv = price_scaler.inverse_transform(train_pred.reshape(-1, 1)).flatten()
            val_pred_inv = price_scaler.inverse_transform(val_pred.reshape(-1, 1)).flatten()
            test_pred_inv = price_scaler.inverse_transform(test_pred.reshape(-1, 1)).flatten()

            def calculate_adj_r2(y_true, y_pred):
                ss_res = np.sum((y_true - y_pred) ** 2)
                ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
                r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
                n = len(y_true)
                adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
                return adj_r2

            train_adj_r2 = calculate_adj_r2(train_y_inv, train_pred_inv)
            val_adj_r2 = calculate_adj_r2(val_y_inv, val_pred_inv)
            test_adj_r2 = calculate_adj_r2(test_y_inv, test_pred_inv)
            overfitting_gap = train_adj_r2 - val_adj_r2

            stock_results[stock_name][model_name]['val_adj_r2'] = val_adj_r2
            stock_results[stock_name][model_name]['test_adj_r2'] = test_adj_r2
            stock_results[stock_name][model_name]['overfitting_gap'] = overfitting_gap
            stock_results[stock_name][model_name]['test_predictions'] = test_pred_inv  # Store predictions
            print(f"\nModel: {model_name} for {stock_name} - Val Adj R²: {val_adj_r2:.6f}, Test Adj R²: {test_adj_r2:.6f}, Overfitting Gap: {overfitting_gap:.6f}")
            print(f"  Training Adj R²: {train_adj_r2:.6f}")
            if overfitting_gap > 0.2:
                print("  Warning: Overfitting detected")
            elif overfitting_gap > 0.1:
                print("  Caution: Moderate overfitting risk")
            else:
                print("  Status: Low overfitting risk")
        except Exception as e:
            print(f"Error analyzing overfitting for {model_name} in {stock_name}: {e}")


Step 6.5: Analyzing overfitting with validation and test sets...

Model: cnn_lstm for BBSE3.SA - Val Adj R²: 0.681381, Test Adj R²: -0.393594, Overfitting Gap: 0.308180
  Training Adj R²: 0.989561

Model: conv1d_lstm for BBSE3.SA - Val Adj R²: 0.839752, Test Adj R²: -18.284177, Overfitting Gap: 0.154225
  Training Adj R²: 0.993977
  Caution: Moderate overfitting risk

Model: bilstm for BBSE3.SA - Val Adj R²: 0.944592, Test Adj R²: 0.304862, Overfitting Gap: 0.053804
  Training Adj R²: 0.998396
  Status: Low overfitting risk

Model: stacked_lstm for BBSE3.SA - Val Adj R²: 0.791241, Test Adj R²: -2.309385, Overfitting Gap: 0.201559
  Training Adj R²: 0.992800

Model: gru for BBSE3.SA - Val Adj R²: 0.968831, Test Adj R²: -0.225989, Overfitting Gap: 0.029989
  Training Adj R²: 0.998819
  Status: Low overfitting risk

Model: lightgbm for BBSE3.SA - Val Adj R²: 0.965605, Test Adj R²: -12.912300, Overfitting Gap: 0.034320
  Training Adj R²: 0.999926
  Status: Low overfitting risk

Model: xgb

In [12]:
# Step 7: Evaluate Best Model on Test Set
print("\nStep 7: Evaluating best model on test set...")
best_test_adj_r2 = -float('inf')
best_model_name = None
best_model = None
best_stock_name = None

for stock_name in stock_results.keys():
    results = stock_results[stock_name]
    for model_name, result in results.items():
        if result.get('test_adj_r2', -float('inf')) > best_test_adj_r2 and result.get('overfitting_gap', float('inf')) <= 0.2:
            best_test_adj_r2 = result['test_adj_r2']
            best_model_name = model_name
            best_model = result['best_model']
            best_stock_name = stock_name

if best_model_name is None:
    print("No model with acceptable overfitting gap found. Using best Test Adj R².")
    for stock_name in stock_results.keys():
        results = stock_results[stock_name]
        for model_name, result in results.items():
            if result.get('test_adj_r2', -float('inf')) > best_test_adj_r2:
                best_test_adj_r2 = result['test_adj_r2']
                best_model_name = model_name
                best_model = result['best_model']
                best_stock_name = stock_name

overfitting_gap = stock_results[best_stock_name][best_model_name].get('overfitting_gap', 'N/A')
print(f"Best model: {best_model_name} for {best_stock_name} with Test Adj R²: {best_test_adj_r2:.6f}, Overfitting Gap: {overfitting_gap}")

def evaluate_test_set(model, model_name, test_X, test_y, price_scaler, stock_name, timestamps):
    try:
        if model_name in ['xgboost', 'lightgbm']:
            test_pred = model.predict(test_X[:, :, 0].reshape(-1, SEQ_LENGTH))
        else:
            test_pred = model.predict(test_X, verbose=0).flatten()

        test_y_inv = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
        test_pred_inv = price_scaler.inverse_transform(test_pred.reshape(-1, 1)).flatten()

        def calculate_adj_r2(y_true, y_pred):
            ss_res = np.sum((y_true - y_pred) ** 2)
            ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
            n = len(y_true)
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
            return adj_r2

        test_adj_r2 = calculate_adj_r2(test_y_inv, test_pred_inv)
        test_timestamps = timestamps.iloc[-len(test_y):]
        return test_pred_inv, test_y_inv, test_timestamps, test_adj_r2
    except Exception as e:
        print(f"Prediction failed for {model_name} in {stock_name}: {e}")
        return None, None, None, None


Step 7: Evaluating best model on test set...
Best model: bilstm for BBSE3.SA with Test Adj R²: 0.304862, Overfitting Gap: 0.053804144516836905


In [13]:
# Evaluate best model on test set
test_X = stock_X_deep[best_stock_name][-150:] if best_model_name in ['cnn_lstm', 'conv1d_lstm', 'bilstm', 'stacked_lstm', 'gru', 'transformer', 'tcn'] else stock_X_ml[best_stock_name][-150:]
test_y = stock_y[best_stock_name][-150:]
predictions, actuals, test_timestamps, test_adj_r2 = evaluate_test_set(
    best_model, best_model_name, test_X, test_y, price_scaler, best_stock_name, stock_data[best_stock_name]['Timestamp']
)
if predictions is not None:
    print(f"Best Model Test Adj R²: {test_adj_r2:.6f}")
    print(f"Predictions (first 5): {predictions[:5]}")
    print(f"Actuals (first 5): {actuals[:5]}")
    print(f"Test timestamps (first 5): {test_timestamps[:5]}")
else:
    print(f"Failed to generate test set predictions for {best_model_name}.")

Best Model Test Adj R²: 0.304862
Predictions (first 5): [35.010265 35.009304 35.0087   35.008274 35.007805]
Actuals (first 5): [34.98998706 35.0023547  34.99472011 34.9856223  34.98214724]
Test timestamps (first 5): 8350   2025-06-25 17:25:00+00:00
8351   2025-06-25 17:26:00+00:00
8352   2025-06-25 17:27:00+00:00
8353   2025-06-25 17:28:00+00:00
8354   2025-06-25 17:29:00+00:00
Name: Timestamp, dtype: datetime64[ns, UTC]


In [14]:
# Step 8: Generate Predictions for All Valid Models on Test Set
print("\nStep 8: Generating test set predictions for all models with Test Adj R² > 0.8...")
prediction_results = {}
for stock_name in stock_results.keys():
    valid_models = {}
    for model_name, result in stock_results[stock_name].items():
        if result.get('test_adj_r2', 0) > 0.65 and result.get('overfitting_gap', float('inf')) <= 0.2:
            valid_models[model_name] = result['best_model']
            print(f"Model '{model_name}' for {stock_name} meets criteria (Test Adj R² > 0.8, Overfitting Gap <= 0.2)")

    if not valid_models:
        print(f"No valid models found for stock {stock_name}")
        continue

    predictions_df = pd.DataFrame()
    test_X = stock_X_deep[stock_name][-150:] if any(m in ['cnn_lstm', 'conv1d_lstm', 'bilstm', 'stacked_lstm', 'gru', 'transformer', 'tcn'] for m in valid_models) else stock_X_ml[stock_name][-150:]
    test_y = stock_y[stock_name][-150:]
    test_timestamps = stock_data[stock_name]['Timestamp'].iloc[-150:]

    for model_name, model in valid_models.items():
        print(f"\nGenerating test set predictions for {model_name} in {stock_name}...")
        predictions, actuals, _, test_adj_r2 = evaluate_test_set(
            model, model_name, test_X, test_y, price_scaler, stock_name, stock_data[stock_name]['Timestamp']
        )
        if predictions is not None:
            predictions_df[model_name] = predictions
            print(f"Model {model_name} - Test Adj R²: {test_adj_r2:.6f}")

    predictions_df['Actual'] = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
    predictions_df['Timestamp'] = test_timestamps
    prediction_results[stock_name] = predictions_df

for stock_name, predictions_df in prediction_results.items():
    print(f"\nTest Set Predictions for {stock_name}:")
    print(predictions_df.head())
    filename = f'{stock_name}_test_predictions.csv'
    predictions_df.to_csv(filename, index=False)
    print(f"\nDownloading {filename}...")
    files.download(filename)


Step 8: Generating test set predictions for all models with Test Adj R² > 0.8...
No valid models found for stock BBSE3.SA
